# Image Generation Using Vision Transformers (ViTs)

**University Assignment Notebook**

In this notebook, I build a small image generation model that uses Vision Transformer ideas.  
The goal is not to make a perfect generator, but to show how transformer/ViT components can be used in a simple image generation pipeline.

The model is a lightweight **ViT-style Variational Autoencoder (ViT-VAE)**:

- the encoder divides an image into patches,
- the patches are processed using real transformer encoder layers,
- the model learns a compact latent representation,
- the decoder reconstructs flower images,
- new flower-like images are made by slightly changing latent vectors from real images.

I keep the project small so it can run on Google Colab free GPU and still be understandable for a university assignment.

## 1. Installing and Importing Libraries

First, I install the Kaggle package because the dataset will be downloaded directly from Kaggle.  
This avoids manually uploading the dataset to Colab.

In [ ]:
!pip -q install kaggle

In [ ]:
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision.transforms as transforms
from torchvision.utils import make_grid

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# For reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

## 2. Kaggle API Setup

To use Kaggle datasets in Colab, Kaggle requires an API token called `kaggle.json`.

Steps:
1. Go to your Kaggle account settings.
2. Create a new API token.
3. Upload the downloaded token file when this cell asks for it.

Sometimes Colab renames the file to something like `kaggle (1).json` if it was uploaded before.  
The setup below handles that case by finding any uploaded JSON file that starts with `kaggle`.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

# Colab may rename the file to "kaggle (1).json", "kaggle (2).json", etc.
kaggle_json_candidates = [
    name for name in uploaded.keys()
    if name.lower().startswith("kaggle") and name.lower().endswith(".json")
]

if len(kaggle_json_candidates) == 0:
    raise FileNotFoundError(
        "Please upload your Kaggle API token file. It should be named kaggle.json "
        "or something like kaggle (1).json."
    )

kaggle_json_file = kaggle_json_candidates[0]
print("Using Kaggle token file:", kaggle_json_file)

In [ ]:
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(kaggle_json_file, KAGGLE_DIR / "kaggle.json")
os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)

print("Kaggle API token is ready.")

## 3. Downloading a Small Image Dataset from Kaggle

For this assignment, I use the **Flowers Recognition** dataset from Kaggle.

The dataset has flower categories such as daisy, dandelion, rose, sunflower, and tulip.  
To make the model learn better with limited Colab time, I train on only one flower class: **daisy**.

Using one class keeps the visual patterns more consistent, so the autoencoder does not have to learn several very different flower types at the same time.

Dataset used:

`alxmamaev/flowers-recognition`

In [ ]:
DATA_DIR = Path("/content/flowers_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the dataset with Kaggle API.
# The command is safe to run again because Kaggle will simply download the zip file.
!kaggle datasets download -d alxmamaev/flowers-recognition -p /content/flowers_dataset

# Unzip automatically
zip_files = list(DATA_DIR.glob("*.zip"))
if len(zip_files) == 0:
    raise FileNotFoundError("No zip file was downloaded. Check Kaggle API setup.")

for z in zip_files:
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

print("Dataset downloaded and extracted to:", DATA_DIR)

# Show the main folders so it is easy to check the extraction result
for folder in sorted([p for p in DATA_DIR.rglob("*") if p.is_dir()])[:15]:
    print(folder)

## 4. What Are Vision Transformers?

A Vision Transformer, or ViT, applies the transformer idea to images.

Instead of processing an image with convolution filters only, a ViT usually does this:

1. Split the image into small patches.
2. Flatten each patch into a vector.
3. Project each patch vector into an embedding.
4. Add positional embeddings so the model knows where each patch came from.
5. Pass the sequence of patch embeddings through transformer layers.

In simple words, the model treats an image like a sequence of visual tokens, similar to how a language transformer treats a sentence as a sequence of word tokens.

## 5. How Transformers Can Be Used for Image Generation

Transformers can be used in image generation in several ways:

- **Autoregressive image generation:** the model predicts image tokens one by one.
- **Diffusion models with transformers:** transformer blocks replace or support CNN blocks in denoising.
- **Masked image modeling:** the model learns to fill missing image patches.
- **Autoencoder-style generation:** the model compresses an image and learns how to reconstruct or sample new images.

In this notebook, I use the autoencoder-style approach because it is simpler and easier to run in a student assignment.  
The transformer part is inside the encoder, where image patches are processed by `nn.TransformerEncoder`.

## 6. CNN-Based vs Transformer-Based Image Generation

| Aspect | CNN-Based Generators | Transformer-Based Generators |
|---|---|---|
| Main idea | Use convolution layers to build images from local features | Use token/patch sequences and attention/transformer layers |
| Strength | Very good at local patterns like edges and textures | Good at modeling long-range relationships between image regions |
| Common examples | DCGAN, U-Net diffusion models, VAE decoders | ViT-VAE, image GPT, MaskGIT, DiT |
| Data requirement | Can work well with smaller datasets | Often needs more data and compute |
| Speed | Usually faster and lighter | Can be heavier because attention scales with sequence length |
| Student-level use | Easier to train | Needs simplification to run well on Colab |

For this assignment, I combine both ideas: a transformer-based encoder and a small CNN-style decoder.

## 7. Loading and Preprocessing the Images

The images are resized to **64 × 64** to keep the notebook lightweight.  
I use only the **daisy** folder and limit the dataset to around **900 images**.

Pixel values are normalized to the range `[-1, 1]` using:

```python
transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
```

This matches the final `Tanh` layer in the decoder.

In [ ]:
class ImageFolderNoLabels(Dataset):
    """Simple image dataset that loads image files recursively from one folder."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        extensions = ["*.jpg", "*.jpeg", "*.png"]
        self.image_paths = []
        for ext in extensions:
            self.image_paths.extend(list(self.root_dir.rglob(ext)))

        self.image_paths = sorted(self.image_paths)

        if len(self.image_paths) == 0:
            raise RuntimeError(f"No images found in {self.root_dir}. Check the dataset path.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image


image_size = 64
batch_size = 64
max_images = 900  # around 800-1000 images for faster Colab training
flower_class = "daisy"

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# The Kaggle zip usually extracts to flowers/daisy, but this search makes it more robust.
possible_class_dirs = [p for p in DATA_DIR.rglob(flower_class) if p.is_dir()]
if len(possible_class_dirs) == 0:
    raise FileNotFoundError(f"Could not find a folder named '{flower_class}' inside {DATA_DIR}")

class_dir = possible_class_dirs[0]
print("Using flower class folder:", class_dir)

dataset = ImageFolderNoLabels(class_dir, transform=transform)
print("Total daisy images found:", len(dataset))

# Use a limited random subset for quick training
subset_size = min(max_images, len(dataset))
indices = list(range(len(dataset)))
random.shuffle(indices)
train_dataset = Subset(dataset, indices[:subset_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Training images used:", len(train_dataset))

## 8. Visualizing a Few Training Images

Before training, it is useful to check that the dataset loaded correctly.

In [ ]:
def denormalize(x):
    """Convert image tensor from [-1, 1] back to [0, 1] for plotting."""
    return (x * 0.5) + 0.5

images = next(iter(train_loader))[:16]
grid = make_grid(denormalize(images), nrow=4)

plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Sample Training Images")
plt.show()

## 9. Model Architecture: A Lightweight ViT-VAE

This model keeps the same autoencoder idea, but the transformer encoder is slightly stronger than before.

Main settings used:

- `LATENT_DIM = 256`
- `nhead = 4`
- `num_layers = 4`
- `dim_feedforward = 512`

The model still clearly uses actual ViT/transformer parts:

```python
nn.TransformerEncoderLayer
nn.TransformerEncoder
```

The decoder is a small convolutional decoder because it is simple and works well for reconstructing image-shaped outputs.

In [ ]:

LATENT_DIM = 256

class PatchEmbedding(nn.Module):
    """Turn an image into patch embeddings."""
    def __init__(self, img_size=64, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()

        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


class ViTAutoencoder(nn.Module):
    """
    Lightweight Vision Transformer autoencoder.
    This keeps the ViT structure simple for a university assignment.
    """

    def __init__(
        self,
        img_size=64,
        patch_size=4,
        in_channels=3,
        embed_dim=128,
        latent_dim=LATENT_DIM,
        num_heads=4,
        num_layers=4,
        ff_dim=512
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size,
            patch_size,
            in_channels,
            embed_dim
        )

        num_patches = self.patch_embed.num_patches

        # ViT-style learnable tokens
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim)
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Latent projection
        self.fc_latent = nn.Linear(embed_dim, latent_dim)

        # Decoder projection
        self.fc_decode = nn.Linear(latent_dim, 256 * 4 * 4)

        # Slightly stronger decoder for sharper images
        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.GELU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.GELU(),

            nn.ConvTranspose2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.GELU(),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.GELU(),

            nn.ConvTranspose2d(32, 16, 4, 2, 1),
            nn.BatchNorm2d(16),
            nn.GELU(),

            nn.Conv2d(16, 3, 3, padding=1),
            nn.Tanh()
        )

        nn.init.normal_(self.pos_embed, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)

    def encode(self, x):

        x = self.patch_embed(x)

        batch_size = x.size(0)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.pos_embed

        x = self.transformer_encoder(x)

        cls_output = x[:, 0]

        latent = self.fc_latent(cls_output)

        return latent

    def decode(self, z):

        x = self.fc_decode(z)

        x = x.view(-1, 256, 4, 4)

        x = self.decoder(x)

        return x

    def forward(self, x):

        latent = self.encode(x)

        reconstructed = self.decode(latent)

        return reconstructed, latent


model = ViTAutoencoder().to(device)

print(model)


## 10. Loss Function

The VAE loss has two parts:

1. **Reconstruction loss:** checks how close the output image is to the input image.
2. **KL divergence loss:** encourages the latent space to stay close to a normal distribution.

I use **L1 loss** instead of MSE because L1 usually gives slightly sharper reconstructions in simple image autoencoders.  
The KL weight is kept small so the model focuses more on reconstructing recognizable flowers.

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=0.0005):
    # L1 loss usually gives slightly sharper reconstructions than MSE for simple image work
    recon_loss = F.l1_loss(recon_x, x, reduction="mean")

    # KL divergence term for VAE
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

## 11. Training Loop

The training loop is kept simple and readable.  
I train for **30 epochs**, which is still reasonable for a small 64×64 dataset on Colab free GPU.

If the notebook needs to run faster during testing, `epochs` can be reduced to 25.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

epochs = 30
loss_history = []
recon_history = []
kl_history = []

model.train()

for epoch in range(epochs):
    total_loss_epoch = 0
    total_recon_epoch = 0
    total_kl_epoch = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss, recon_loss, kl_loss = vae_loss(recon, batch, mu, logvar)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss_epoch += loss.item()
        total_recon_epoch += recon_loss.item()
        total_kl_epoch += kl_loss.item()

    avg_loss = total_loss_epoch / len(train_loader)
    avg_recon = total_recon_epoch / len(train_loader)
    avg_kl = total_kl_epoch / len(train_loader)

    loss_history.append(avg_loss)
    recon_history.append(avg_recon)
    kl_history.append(avg_kl)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.4f} | Recon: {avg_recon:.4f} | KL: {avg_kl:.4f}"
    )

## 12. Loss Visualization

The graph below shows whether the model is learning during training.  
For this kind of simple assignment model, the loss should generally go down.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history, label="Total loss")
plt.plot(recon_history, label="Reconstruction loss")
plt.plot(kl_history, label="KL loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)
plt.show()

## 13. Reconstruction Results

Here I compare original images with reconstructed images.  
The reconstructions show what the model learned after compressing images through the ViT-style encoder.

In [ ]:
model.eval()

with torch.no_grad():
    test_images = next(iter(train_loader))[:8].to(device)
    recon_images, _, _ = model(test_images)

# Show originals on the first row and reconstructions on the second row
comparison = torch.cat([test_images.cpu(), recon_images.cpu()], dim=0)
comparison = denormalize(comparison).clamp(0, 1)
grid = make_grid(comparison, nrow=8)

plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Top row: Original daisy images | Bottom row: Reconstructed images")
plt.show()

## 14. Generating New Images

Fully random latent vectors often produce noisy images in this small model.  
A better student-level approach is:

1. take real flower images,
2. encode them into latent vectors,
3. add a small amount of Gaussian noise,
4. decode the changed latent vectors.

This still demonstrates generation because the output is not just a direct reconstruction, but it stays close to the learned flower image space.

In [ ]:

# Generate images by slightly modifying real latent vectors
# This produces more realistic flower-like outputs.

model.eval()

with torch.no_grad():

    real_images, _ = next(iter(train_loader))

    real_images = real_images[:16].to(device)

    latent_vectors = model.encode(real_images)

    noise = torch.randn_like(latent_vectors) * 0.08

    modified_latents = latent_vectors + noise

    generated_images = model.decode(modified_latents)

    generated_images = generated_images.cpu()

# Denormalize for display
generated_images = (generated_images * 0.5) + 0.5
generated_images = torch.clamp(generated_images, 0, 1)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))

for i, ax in enumerate(axes.flat):

    img = generated_images[i].permute(1, 2, 0).numpy()

    ax.imshow(img)

    ax.axis("off")

plt.suptitle("Generated flower-like images using noisy latent vectors")

plt.tight_layout()

plt.show()


## 15. Why the Results Are Still Not Perfect

The generated images should look more flower-like than using completely random latent vectors, but they may still be somewhat blurry.

This is expected because:

- the model is intentionally lightweight,
- the images are only 64×64,
- the dataset subset is small,
- training is limited to Colab-friendly time,
- this is a simple ViT autoencoder, not a large diffusion model or GAN.

So the goal here is not photorealistic flower generation.  
The main goal is to clearly demonstrate how ViT-style patch embeddings and transformer encoder layers can be used inside an image generation project.

Because this is still a lightweight student-level Vision Transformer autoencoder trained on a small flower dataset and limited GPU time, the generated images may still look somewhat soft or blurry. Vision Transformers usually need larger datasets and longer training to produce highly detailed images.

## 16. Advantages and Drawbacks of ViTs for Image Generation

### Advantages

- ViTs can model relationships between distant image regions.
- They are flexible because images can be represented as patch tokens.
- Transformer blocks are now used in many modern generative models.
- They can scale well when large datasets and strong hardware are available.

### Drawbacks

- They usually need more data than CNNs.
- Attention can become expensive when the number of patches is large.
- Small transformer image generators may produce blurry or weak images.
- Training high-quality transformer generators is usually harder than training a simple CNN autoencoder.

In this notebook, these drawbacks are visible because the model is intentionally small and trained with limited compute.

Because this is still a lightweight student-level Vision Transformer autoencoder trained on a small flower dataset and limited GPU time, the generated images may still look somewhat soft or blurry. Vision Transformers usually need larger datasets and longer training to produce highly detailed images.

## 17. Conclusion

In this notebook, I built a small image generation model using a Vision Transformer-style encoder.  
The model is not meant to compete with advanced image generators, but it demonstrates the main concept clearly:

- images are split into patches,
- patch tokens are processed using transformer encoder layers,
- the latent representation is decoded into images,
- new images are generated by adding small noise to encoded real image latents.

Using one flower class, L1 reconstruction loss, `Tanh` output, and a slightly stronger transformer encoder helps the results compared with a weaker setup.  
The outputs can still be blurry, but that is normal for a lightweight student-level ViT autoencoder trained on limited data.

Because this is still a lightweight student-level Vision Transformer autoencoder trained on a small flower dataset and limited GPU time, the generated images may still look somewhat soft or blurry. Vision Transformers usually need larger datasets and longer training to produce highly detailed images.

## References

Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., Dehghani, M., Minderer, M., Heigold, G., Gelly, S., Uszkoreit, J., & Houlsby, N. (2021). *An image is worth 16x16 words: Transformers for image recognition at scale*. International Conference on Learning Representations.

Kingma, D. P., & Welling, M. (2014). *Auto-encoding variational Bayes*. International Conference on Learning Representations.

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention is all you need*. Advances in Neural Information Processing Systems, 30.

Mamaev, A. (2018). *Flowers Recognition* [Data set]. Kaggle. https://www.kaggle.com/datasets/alxmamaev/flowers-recognition